# Task 5: LoRA Fine-Tuning

Trains `adapter_baroque` and `adapter_romantic` on cleaned MAESTRO windows.

The base model (stanford-crfm/music-medium-800k) was pretrained on **Lakh MIDI**, not MAESTRO.

**Prerequisites**: smoke test passed, MAESTRO filtered (manifest.csv exists).

In [ ]:
# Cell 1: Mount Drive and clone the active branch
from google.colab import drive
drive.mount('/content/drive')

import subprocess
import os
repo_dir = '/content/thesis'
repo_url = 'https://github.com/faraz-irshad/music-style-transformer.git'
if os.path.isdir(repo_dir):
    subprocess.run(['git', '-C', repo_dir, 'fetch', 'origin', 'fixes-and-ui'], check=True)
    subprocess.run(['git', '-C', repo_dir, 'checkout', '-B', 'fixes-and-ui', 'origin/fixes-and-ui'], check=True)
else:
    subprocess.run(['git', 'clone', '-b', 'fixes-and-ui', '--single-branch', repo_url, repo_dir], check=True)
os.chdir(repo_dir)
commit_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print(f'Using fixes-and-ui commit {commit_hash}')

In [ ]:
# Cell 2: Install dependencies
exec(open('colab/setup_cell.py').read())

In [ ]:
# Cell 3: Config — edit these paths
import yaml
with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['maestro_csv'] = '/content/maestro-v3.0.0/maestro-v3.0.0.csv'
cfg['data']['maestro_midi_dir'] = '/content/maestro-v3.0.0/'
cfg['data']['output_dir'] = '/content/thesis_data'
cfg['training']['drive_dir'] = '/content/drive/MyDrive/thesis_adapters'

with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f)
print('Config updated.')

In [ ]:
# Cell 4: Dry run (~2 min) — validate pipeline before full training
!python -u train_lora.py --config config.yaml --style baroque --dry_run

In [ ]:
# Cell 5: Train Baroque adapter
!python -u train_lora.py --config config.yaml --style baroque

In [ ]:
# Cell 6: Train Romantic adapter
!python -u train_lora.py --config config.yaml --style romantic

In [ ]:
# Cell 7: Plot training curves
import json, matplotlib.pyplot as plt
from pathlib import Path

drive_dir = cfg['training']['drive_dir']
experiment_root = Path(cfg['training'].get('experiment_dir') or
                       Path(drive_dir) / 'experiments' /
                       cfg['training'].get('experiment_name', 'neural_alignment_v1'))
plot_path = experiment_root / 'training_curves.png'
if plot_path.exists():
    raise FileExistsError(f'Refusing to overwrite {plot_path}')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, style in zip(axes, ('baroque', 'romantic')):
    log_path = experiment_root / f'adapter_{style}' / 'train_log.jsonl'
    if not log_path.exists():
        ax.set_title(f'{style} — no log')
        continue
    steps, train_losses = [], []
    unconditional_losses, conditioned_losses, val_steps = [], [], []
    with open(log_path) as f:
        for line in f:
            entry = json.loads(line)
            steps.append(entry['step'])
            train_losses.append(entry['train_loss'])
            if 'val_unconditional_piano_loss' in entry:
                val_steps.append(entry['step'])
                unconditional_losses.append(entry['val_unconditional_piano_loss'])
                conditioned_losses.append(entry['val_conditioned_accompaniment_loss'])
    ax.plot(steps, train_losses, label='train')
    if unconditional_losses:
        ax.plot(val_steps, unconditional_losses, label='val unconditional piano', marker='o')
        ax.plot(val_steps, conditioned_losses, label='val conditioned accompaniment', marker='o')
    ax.set_title(f'{style} adapter')
    ax.set_xlabel('step'); ax.set_ylabel('loss'); ax.legend()

plt.tight_layout()
plt.savefig(plot_path, dpi=120)
plt.show()